[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Transactions and Errors


## What you will be able to do

Say what happens to a connection after one statement fails, and why the statement after it fails too
with an error about a transaction rather than about itself. End the transaction and carry on. Read a
failure for what it says: its class, its SQLSTATE code, and the constraint it names. Turn a SQLSTATE
string into the class that matches it. Put a savepoint around one risky write so a bad row costs you
that row rather than the batch. Leave a block without letting an exception escape. And find the same
codes in asyncpg, where the classes have different names and carry the same numbers.


## The idea

### The problem

One statement fails, you catch the exception, and the next statement fails too, with a message about
the transaction being aborted rather than anything to do with what you asked. Nothing you do on that
connection works until the transaction ends.

That is not a driver quirk. PostgreSQL puts a transaction into an aborted state the moment any
statement in it fails, and from then on it refuses everything except a rollback. The **Peewee, Deep
Dive** guide describes this difference from the outside, as the thing that makes SQLite and
PostgreSQL behave differently when an error is swallowed. This is the notebook that shows it.

### What a transaction is here

The connection has one, whether or not you asked for one. psycopg opens a transaction on the first
statement and leaves it open until you commit, roll back, or close the connection. That is why the
`with` block in **Connecting and Executing** was able to commit something you never told it to
commit.

So there are three states a connection can be in: no transaction yet, a transaction that is fine,
and a transaction that is aborted. The third one is what this notebook is about.

### Why it works that way

PostgreSQL cannot know what your next statement assumes. If an `INSERT` failed and you carry on as
though it worked, everything after it is built on something that is not there. Refusing the rest is
the only safe thing to do without guessing, so the server stops and makes you say what you meant, by
rolling back or by going back to a savepoint.

### Where this shows up

Any loop that writes rows and tries to keep going past a bad one. Any request handler that catches a
database error, logs it, and then tries to write an audit row on the same connection. Both are the
same mistake, and the fix for both is a savepoint.

### What this notebook covers

The aborted transaction and how to get out of it. `autocommit`, and the statement that cannot run in
a transaction at all. Reading an error: the class, `sqlstate`, and `diag`. Looking a code up.
Savepoints through `conn.transaction()`, which is the same object **Connecting and Executing** used
to keep a connection open. `psycopg.Rollback`. The same failures in asyncpg. Then the four errors,
three of which are the ones the worked examples were built on.

Isolation levels, serialization failures and the retry loop that goes with them are not here. They
are a subject of their own, they are the same in both drivers, and the PostgreSQL documentation on
transaction isolation is the right place for them.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg

with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT 1/0")
    except psycopg.errors.DivisionByZero as error:
        print("SELECT 1/0 ->", error)

    try:
        conn.execute("SELECT 1")                            # nothing is wrong with this one
    except psycopg.errors.InFailedSqlTransaction as error:
        print("SELECT 1   ->", error)
```

```
SELECT 1/0 -> division by zero
SELECT 1   -> current transaction is aborted, commands ignored until end of transaction block
```

`SELECT 1` is not a statement that can fail. It failed because the connection was carrying an
aborted transaction, and it would have gone on failing for every statement after it until the
transaction ended.


## Setup

Nine imports, both drivers, the server, and two small tables.

- `psycopg` and `asyncpg` are the drivers, and `errors` is psycopg's module of exception classes,
  one per SQLSTATE condition
- `subprocess`, `sys`, `os`, `getpass` and `time` stand the server up, which **A Server of Your Own**
  takes apart
- `version` and `PackageNotFoundError` install the drivers where they are missing

`build_tables` makes a `parent` with one row and a `child` that points at it, so this notebook has a
primary key to violate and a foreign key to violate. `ids` says what is in `child` at any moment,
which is how each example below shows what survived.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def build_tables():
    """A parent and a child, so this notebook has both kinds of constraint to break."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS child, parent CASCADE")
        conn.execute("CREATE TABLE parent (id int PRIMARY KEY)")
        conn.execute("INSERT INTO parent VALUES (1)")
        conn.execute("CREATE TABLE child (id int PRIMARY KEY, "
                     "parent_id int REFERENCES parent (id))")


def ids():
    """Whatever is in child now, as a list, so a cell can show what survived."""
    with psycopg.connect("dbname=guide") as conn:
        return [row[0] for row in conn.execute("SELECT id FROM child ORDER BY id")]


print("server:", start_server())
print(report())
build_tables()
print("parent and child are ready | child holds:", ids())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
parent and child are ready | child holds: []


## Worked examples

### The aborted transaction, and the way out

The first look showed the failure. Here is the state it leaves behind, asked for directly:


In [2]:
conn = psycopg.connect("dbname=guide")
print("before anything:", conn.info.transaction_status.name)

conn.execute("SELECT 1")
print("after a statement:", conn.info.transaction_status.name)

try:
    conn.execute("SELECT 1/0")
except errors.DivisionByZero:
    pass
print("after a failure:  ", conn.info.transaction_status.name)


before anything: IDLE
after a statement: INTRANS
after a failure:   INERROR


`IDLE` is no transaction, `INTRANS` is one that is fine, and `INERROR` is one that will refuse
everything. Those three names are worth knowing because they turn a confusing error into a question
you can ask.

Getting out is a rollback, and then the connection is ordinary again:


In [3]:
conn.rollback()
print("after a rollback:", conn.info.transaction_status.name)
print("and it works:    ", conn.execute("SELECT 1").fetchone())
conn.close()


after a rollback: IDLE
and it works:     (1,)


### The statement that cannot be in a transaction

`CREATE DATABASE` is the one everybody meets first, because psycopg has a transaction open before you
get a chance to think about it:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("CREATE DATABASE nope")
    except errors.ActiveSqlTransaction as error:
        print(type(error).__module__ + "." + type(error).__name__ + ":", error)


psycopg.errors.ActiveSqlTransaction: CREATE DATABASE cannot run inside a transaction block


`autocommit=True` is the answer, and it means what it says: every statement is its own transaction,
committed as it finishes. It is not a way of turning transactions off, it is a way of having one per
statement.


In [5]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    print("transaction status:", conn.info.transaction_status.name)
    conn.execute("CREATE DATABASE nope")
    print("made it")
    conn.execute("DROP DATABASE nope")
    print("and dropped it again")


transaction status: IDLE
made it
and dropped it again


`CREATE INDEX CONCURRENTLY`, `VACUUM` and `REINDEX` have the same rule, and the same answer.

### Reading a failure

An exception from PostgreSQL carries more than its message. The class is chosen from the SQLSTATE
code, and `diag` holds the fields the server sent:


In [6]:
build_tables()

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("INSERT INTO child VALUES (1, 1)")
    try:
        conn.execute("INSERT INTO child VALUES (1, 1)")
    except errors.UniqueViolation as error:
        print("class:          ", type(error).__name__)
        print("sqlstate:       ", error.sqlstate)
        print("constraint_name:", error.diag.constraint_name)
        print("table_name:     ", error.diag.table_name)
        print("message:        ", error)


class:           UniqueViolation
sqlstate:        23505
constraint_name: child_pkey
table_name:      child
message:         duplicate key value violates unique constraint "child_pkey"
DETAIL:  Key (id)=(1) already exists.


The constraint name is the useful part. A message saying a unique constraint was violated is not
actionable on its own, and `child_pkey` tells you which one, which is how a handler can tell "this
id is taken" from "this email is taken" without reading English.

A different constraint gives a different class and a different code:


In [7]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    try:
        conn.execute("INSERT INTO child VALUES (2, 999)")           # there is no parent 999
    except errors.ForeignKeyViolation as error:
        print("class:   ", type(error).__name__, "| sqlstate:", error.sqlstate)
        print("constraint_name:", error.diag.constraint_name)
        print("message: ", error)


class:    ForeignKeyViolation | sqlstate: 23503
constraint_name: child_parent_id_fkey
message:  insert or update on table "child" violates foreign key constraint "child_parent_id_fkey"
DETAIL:  Key (parent_id)=(999) is not present in table "parent".


Both are subclasses of `IntegrityError`, so code that does not care which one can catch that instead.
The codes themselves are PostgreSQL's, not psycopg's, which is why they appear unchanged in asyncpg
further down.

### Turning a code into a class

`errors.lookup` goes from the string the server sends to the class psycopg would have raised:


In [8]:
print("23505 ->", errors.lookup("23505").__name__)
print("23503 ->", errors.lookup("23503").__name__)
print("25P02 ->", errors.lookup("25P02").__name__)

try:
    errors.lookup("ZZ999")
except KeyError as error:
    print("a code with no class ->", type(error).__name__ + ":", error)


23505 -> UniqueViolation
23503 -> ForeignKeyViolation
25P02 -> InFailedSqlTransaction
a code with no class -> KeyError: 'ZZ999'


That is useful in the direction you meet it: a log or an alert has the five character code in it,
and this is how you find out what it was without searching. It is also how you catch a condition
psycopg has no name for, by looking it up and catching whatever comes back.

### A savepoint, so one bad row costs one row

This is the fix for the loop that has to keep going. `conn.transaction()` inside an open transaction
is a savepoint, and rolling back to it undoes only what happened inside the block:


In [9]:
build_tables()

with psycopg.connect("dbname=guide") as conn:
    kept, skipped = [], []
    for number in (2, 3, 3, 4):
        try:
            with conn.transaction():                                # a savepoint, one row wide
                conn.execute("INSERT INTO child VALUES (%s, 1)", (number,))
            kept.append(number)
        except errors.UniqueViolation:
            skipped.append(number)

print("kept:", kept, "| skipped:", skipped)
print("rows:", ids())


kept: [2, 3, 4] | skipped: [3]
rows: [2, 3, 4]


Three rows in, one skipped, and the transaction was never poisoned. Without the savepoint the same
loop loses everything, because the failure aborts the whole transaction and the rows written before
it go with it:


In [10]:
build_tables()

with psycopg.connect("dbname=guide") as conn:
    try:
        for number in (5, 6, 6, 7):
            conn.execute("INSERT INTO child VALUES (%s, 1)", (number,))
    except errors.UniqueViolation:
        print("stopped at the duplicate")

    try:
        conn.execute("SELECT count(*) FROM child")
    except errors.InFailedSqlTransaction:
        print("and the connection is aborted, so even counting fails")

print("rows:", ids(), "<- 5 and 6 were written, and did not survive")


stopped at the duplicate
and the connection is aborted, so even counting fails
rows: [] <- 5 and 6 were written, and did not survive


### Leaving a block without an exception

Sometimes the code decides to undo a block for a reason that is not an error. `psycopg.Rollback`
does that: raised inside a `transaction()` block, it rolls the block back and stops there rather
than travelling up:


In [11]:
build_tables()

with psycopg.connect("dbname=guide") as conn:
    with conn.transaction():
        conn.execute("INSERT INTO child VALUES (8, 1)")             # kept

    with conn.transaction():
        conn.execute("INSERT INTO child VALUES (9, 1)")
        raise psycopg.Rollback                                      # undone, and nothing escapes

    print("the code after the block runs:", True)

print("rows:", ids())


the code after the block runs: True
rows: [8]


No `try` around it, and no exception reaches this cell. That is the difference between `Rollback` and
raising something of your own: an ordinary exception would also undo the block, and would then keep
going up until something caught it.

### The same failures in asyncpg

Different classes, same codes. The name is the PostgreSQL condition with `Error` on the end:


In [12]:
build_tables()
conn = await asyncpg.connect(database="guide")
await conn.execute("INSERT INTO child VALUES (1, 1)")

try:
    await conn.execute("INSERT INTO child VALUES (1, 1)")
except asyncpg.exceptions.UniqueViolationError as error:
    print("class:   ", type(error).__module__ + "." + type(error).__name__)
    print("sqlstate:", error.sqlstate, "| constraint:", error.constraint_name)
    print("detail:  ", error.detail)
await conn.close()


class:    asyncpg.exceptions.UniqueViolationError
sqlstate: 23505 | constraint: child_pkey
detail:   Key (id)=(1) already exists.


`23505` in both drivers, because it is the server's number. `detail` is a field psycopg puts on
`diag` and asyncpg puts on the exception itself, which is the kind of difference that makes porting
code between them tedious rather than hard.

asyncpg's transaction block is spelled `async with conn.transaction()`, and it nests into savepoints
the same way psycopg's does. **asyncpg** is the notebook that takes its transaction model apart,
because it differs in a way this one does not prepare you for: a statement outside a block is
committed as it runs.

### When to reach for which

| What you want | How to write it |
|---|---|
| one transaction around several writes | `with conn.transaction():` |
| one transaction per statement | `psycopg.connect(..., autocommit=True)` |
| a statement that cannot be in a transaction | `autocommit=True`, then run it |
| one bad row not to cost the batch | a nested `with conn.transaction():` per row |
| to undo a block on purpose | `raise psycopg.Rollback` inside it |
| to know why a write was refused | `error.sqlstate` and `error.diag.constraint_name` |
| the class for a code you have | `errors.lookup("23505")` |
| to catch any constraint violation | `except errors.IntegrityError` |
| the database to settle a duplicate | `INSERT ... ON CONFLICT DO NOTHING` |

The default is one transaction around a unit of work, and a savepoint inside it only where you
intend to carry on after a failure. Reach for `autocommit` for statements that demand it and for
read-only work where a long transaction would hold things open for no reason.

### An import that survives its bad rows, finished

Everything above, as the loop the savepoint was for: it writes what it can, reports what it could
not, and tells the caller why each one was refused.


In [13]:
def load(rows):
    """Write what is writable, and say what happened to the rest."""
    written, refused = [], []
    with psycopg.connect("dbname=guide") as conn:
        for identifier, parent in rows:
            try:
                with conn.transaction():                            # this row, on its own
                    conn.execute("INSERT INTO child VALUES (%s, %s)", (identifier, parent))
                written.append(identifier)
            except errors.IntegrityError as error:
                refused.append((identifier, type(error).__name__, error.diag.constraint_name))
    return written, refused


build_tables()
written, refused = load([(1, 1), (2, 1), (2, 1), (3, 999), (4, 1)])

print("written:", written)
for identifier, kind, constraint in refused:
    print(f"  refused {identifier}: {kind} on {constraint}")
print("rows:", ids())


written: [1, 2, 4]
  refused 2: UniqueViolation on child_pkey
  refused 3: ForeignKeyViolation on child_parent_id_fkey
rows: [1, 2, 4]


One duplicate and one missing parent, both refused for reasons the caller can act on, and the three
good rows committed. The `IntegrityError` catch covers both conditions, and the class name in the
report says which one each row hit.

### Where each part came from

| In the import | What it relies on | The section that showed it |
|---|---|---|
| `with conn.transaction():` per row | a savepoint one row wide | A savepoint |
| `except errors.IntegrityError` | one catch for both constraint kinds | Reading a failure |
| `error.diag.constraint_name` | the server naming what was violated | Reading a failure |
| the outer `with psycopg.connect(...)` | one transaction, committed at the end | **Connecting and Executing** |
| carrying on after a refusal | the transaction never entering the aborted state | The aborted transaction |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/03-transactions-and-errors-solutions.ipynb).

**1.** Fail a statement on purpose, print the connection's transaction status, then get the
connection working again and print the status a second time.


In [14]:
# your code here


**2.** Catch a unique violation and print its class, its SQLSTATE code and the constraint it names.


In [15]:
# your code here


**3.** Write four rows where the third is a duplicate, with a savepoint around each, and print which
were kept.


In [16]:
# your code here


**4.** Do the same without the savepoints and show how many rows survive.


In [17]:
# your code here


**5.** Look up the classes for `23505` and `23503`, and show what `errors.lookup` does with a code
that has no class.


In [18]:
# your code here


**6.** Cause the same duplicate through asyncpg and print the SQLSTATE code, to show it is the same
number as psycopg gave.


In [19]:
# your code here


## Common errors

### psycopg.errors.InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block


In [20]:
with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT 1/0")
    except errors.DivisionByZero:
        pass

    conn.execute("SELECT count(*) FROM child")


InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block

The statement in the traceback is fine. The connection is not, and it has been refusing everything
since the division by zero several lines earlier, which is what makes this error confusing the first
time: it points at the wrong line by design, because every line after the real failure gets the same
answer.

Two ways out, and which one is right depends on whether you meant to carry on:


In [21]:
with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT 1/0")
    except errors.DivisionByZero:
        conn.rollback()                                             # give up on the work so far
    print("after a rollback:", conn.execute("SELECT count(*) FROM child").fetchone())

with psycopg.connect("dbname=guide") as conn:
    try:
        with conn.transaction():                                    # or keep the failure inside a block
            conn.execute("SELECT 1/0")
    except errors.DivisionByZero:
        pass
    print("after a savepoint: ", conn.execute("SELECT count(*) FROM child").fetchone())


after a rollback: (3,)
after a savepoint:  (3,)


### psycopg.errors.ActiveSqlTransaction: CREATE DATABASE cannot run inside a transaction block


In [22]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("CREATE DATABASE reports")


ActiveSqlTransaction: CREATE DATABASE cannot run inside a transaction block

psycopg opened a transaction on the first statement, which for this connection was this one. There is
no way to write this statement so that it works on a connection in a transaction, so the fix is a
connection that does not use them.

`autocommit=True` is per connection, so the usual shape is a short connection for the statement that
needs it:


In [23]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("CREATE DATABASE reports")
    print("created:", conn.execute("SELECT 1 FROM pg_database WHERE datname = 'reports'").fetchone())
    conn.execute("DROP DATABASE reports")


created: (1,)


### psycopg.errors.UniqueViolation: duplicate key value violates unique constraint "child_pkey"


In [24]:
build_tables()
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("INSERT INTO child VALUES (1, 1)")
    conn.execute("INSERT INTO child VALUES (1, 1)")


UniqueViolation: duplicate key value violates unique constraint "child_pkey"
DETAIL:  Key (id)=(1) already exists.

The row is already there. This is the most common write failure there is, and the thing worth
knowing is that you have three different answers to it depending on what you meant.

Catch it, if a duplicate is an event your code should report. Use `ON CONFLICT DO NOTHING`, if a
duplicate means "already done" and there is nothing to say. Use `ON CONFLICT ... DO UPDATE`, if the
new row should win:


In [25]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    cur = conn.execute("INSERT INTO child VALUES (1, 1) ON CONFLICT DO NOTHING")
    print("DO NOTHING wrote", cur.rowcount, "rows, and raised nothing")

    conn.execute("INSERT INTO child VALUES (1, 1) "
                 "ON CONFLICT (id) DO UPDATE SET parent_id = EXCLUDED.parent_id")
    print("DO UPDATE left the table with", len(ids()), "row")


DO NOTHING wrote 0 rows, and raised nothing
DO UPDATE left the table with 1 row


### psycopg.errors.ForeignKeyViolation: insert or update on table "child" violates foreign key constraint "child_parent_id_fkey"


In [26]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("INSERT INTO child VALUES (2, 999)")


ForeignKeyViolation: insert or update on table "child" violates foreign key constraint "child_parent_id_fkey"
DETAIL:  Key (parent_id)=(999) is not present in table "parent".

There is no parent with that id, so the row cannot exist without breaking the rule the table was
declared with. Unlike the unique violation, this one usually means the code has the wrong value
rather than that the work is already done, so catching it and carrying on is rarely right.

The **Relationships** notebook of the **Peewee, Deep Dive** guide had to switch this enforcement on
with a pragma, because SQLite does not check by default. PostgreSQL has no such setting: the
constraint is checked because it exists.


In [27]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("INSERT INTO parent VALUES (999)")                 # make the parent first
    conn.execute("INSERT INTO child VALUES (2, 999)")
    print("with a parent that exists:", ids())


with a parent that exists: [1, 2]


## Recap

- PostgreSQL aborts a transaction as soon as any statement in it fails, and then refuses every
  statement until the transaction ends. The error you see names the transaction, not the statement.
- `conn.info.transaction_status` is `IDLE`, `INTRANS` or `INERROR`, which is how you ask which of the
  three states a connection is in.
- `autocommit=True` gives one transaction per statement, which is what `CREATE DATABASE`, `VACUUM`
  and `CREATE INDEX CONCURRENTLY` need.
- A failure carries its SQLSTATE code in `sqlstate` and the server's fields in `diag`, of which
  `constraint_name` is usually the one worth acting on.
- `errors.lookup("23505")` turns a code into the class, and raises `KeyError` for a code with no
  class.
- `with conn.transaction():` inside an open transaction is a savepoint. One per risky write is what
  lets a loop carry on, and without it a single bad row loses everything written before it.
- `raise psycopg.Rollback` undoes a block without the exception escaping it.
- asyncpg raises differently named classes carrying the same SQLSTATE codes, with `detail` and
  `constraint_name` on the exception rather than on a `diag` object.


## What is next

The **Placeholders and Identifiers** notebook is about the values in a query: `%s` and why the
f-strings this guide has been writing are a habit worth breaking, the list that needs `= ANY` rather
than `IN`, `executemany`, and the table name no placeholder can carry, which needs `sql.Identifier`
instead.


---

&#8592; **Previous:** [Connecting and Executing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/02-connecting-and-executing.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Placeholders and Identifiers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/04-placeholders-and-identifiers.ipynb) &#8594;
